In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import rcParams
import seaborn as sns
%matplotlib inline 
import plotly.express as px
import plotly.graph_objs as go
import plotly.figure_factory as ff
from plotly import tools
from plotly.offline import download_plotlyjs, init_notebook_mode, plot, iplot

init_notebook_mode(connected=True)

import re
import random

from sklearn.datasets import make_classification
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler
from imblearn.over_sampling import SMOTE
from sklearn.utils.class_weight import compute_class_weight

import gc
from datetime import datetime 
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score, classification_report, recall_score, precision_score, f1_score, matthews_corrcoef, average_precision_score
import sklearn.metrics as metrics
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from catboost import CatBoostClassifier

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.neighbors import KNeighborsClassifier

import lightgbm as lgb
from lightgbm import LGBMClassifier
import xgboost as xgb
from sklearn import svm

import shap

import lime
import lime.lime_tabular

import category_encoders as ce

from IPython.display import HTML
import warnings

import string


pd.set_option('display.max_columns', 100)


RFC_METRIC = 'gini'  #metric used for RandomForrestClassifier
NUM_ESTIMATORS = 100 #number of estimators used for RandomForrestClassifier
NO_JOBS = 4 #number of parallel jobs used for RandomForrestClassifier


#TRAIN/VALIDATION/TEST SPLIT
#VALIDATION
VALID_SIZE = 0.20 # simple validation using train_test_split
TEST_SIZE = 0.20 # test size using_train_test_split

#CROSS-VALIDATION
NUMBER_KFOLDS = 5 #number of KFolds for cross-validation



RANDOM_STATE = 1200

# D6 (Revision_Plan_V02.md): "temporal" is the adopted primary design -- a
# chronological 64/16/20 cut on trans_time. "random" reproduces the submitted
# shuffled split and is kept only to quantify the leakage it introduces.
SPLIT_MODE = "temporal"

# B4 (Revision_Plan_V03.md) -- DECIDED 2026-09-14: drop it.
# Under a chronological split, trans_time ("Time difference from the first
# transaction") is effectively a row index: every validation and test row lies
# beyond the largest value seen in training. Dropping it lifted the untuned
# Decision Tree from AUC-PR 0.463 to 0.570 and left the boosters unchanged
# (split_comparison_V01.csv). trans_time_grp, which is cyclical, is kept either
# way. The D8 hyperparameters were tuned WITH trans_time; if this is set to
# True, re-run revision_scripts/d8_tuning.py without it before reporting --
# done: the configurations below come from d8_tuning.py --drop-trans-time.
DROP_TRANS_TIME = True

# D10 (Revision_Plan_V03.md, decision 3): add the seven client-level behavioural
# features built by MLProj01_fraudTrain_clientfeatures_V01.ipynb. The paper reports
# both configurations -- encoding only, and encoding + client features -- so run this
# notebook once with each setting.
CLIENT_FEATURES = False

# J1 / Phase 7 (Revision_Phase7_Plan_V01.md): the conventional baseline -- the raw Sparkov
# columns under conventional handling (revision_scripts/raw_baseline_features.py) -- instead of
# the encoded features. Four runs of record: RAW_BASELINE x CLIENT_FEATURES.
RAW_BASELINE = False

# V26 (Phase 7): every configuration runs on the CORRECTED encoded files (encoding_V25 /
# revision_scripts/rebuild_encoded.py) with the V03 hyperparameter tables, so CONFIG and TAG
# carry "p7" and nothing written by V25 is overwritten. CONFIG selects the tuned
# hyperparameters; TAG names every output file. They match revision_scripts/ablation_run.py's
# p7_* configurations, whose --check compares against this notebook's test tables.
assert DROP_TRANS_TIME, "Phase 7 configurations all drop trans_time"
CONFIG = "p7_notime" + ("_raw" if RAW_BASELINE else "") + ("_client" if CLIENT_FEATURES else "")
TAG = SPLIT_MODE + "_" + CONFIG

# D9: fraud-investigation budget. The reported operating point is the
# threshold that alerts this share of transactions.
ALERT_RATE = 0.005

MAX_ROUNDS = 1000 #lgb iterations
EARLY_STOP = 50 #lgb early stop 
OPT_ROUNDS = 1000  #To be adjusted based on best validation rounds
VERBOSE_EVAL = 50 #Print out metric result

IS_LOCAL = False

warnings.filterwarnings('default') # enable Python warnings in this file

In [ ]:
import os
from pathlib import Path

# Run from the project directory, the one containing fraudTrain_encoded.csv.
path = str(Path.cwd()) + os.sep
if not (Path(path) / "fraudTrain_encoded.csv").exists():
    print("fraudTrain_encoded.csv not found: start Jupyter in the project directory")
os.chdir(path)
print("working directory:", path)


#### Read CSV and Check

In [ ]:
# V26: four data files -- encoded or raw baseline, with or without the client features
DATA_FILE = ("fraudTrain_raw_baseline" if RAW_BASELINE else "fraudTrain_encoded") + ("_client" if CLIENT_FEATURES else "") + ".csv"
df_enc = pd.read_csv(path + DATA_FILE)
print(DATA_FILE, df_enc.shape)

df_enc


In [ ]:
df_enc.loc[(df_enc['trans_time']>=6613) & (df_enc['trans_time']<=9427)]

In [ ]:
column_names_dict = {
'trans_time': 'Time difference from the first transaction',
'trans_time_grp': 'Transaction time group',
'category_1': 'Offline vs. online',
'category_2': 'Non-essential vs. essential',
'category_3': 'Average item cost',
'category_4a': 'Purpose of purchase: household/wellbeing',
'category_4b': 'Purpose of purchase: entertainment/lifestyle',
'category_4c': 'Purpose of purchase: shopping-related',
'category_interaction_1': 'Interaction 1: Non-essential vs. essential*Average item cost',
'category_interaction_2': 'Interaction 2: Non-essential vs. essential*Purpose of purchase: household or wellbeing',
'amt': 'Amount spent (USD)',
'gender': 'Gender',
'city_1': 'City socioeconomic status',
'city_2': 'City social inequality level',
'city_3': 'City Rural-Urban',
'city_4a': 'City high-speed fixed broadband coverage',
'city_4b': 'City high-speed mobile broadband',
'state_x': 'State West vs East',
'state_y': 'State North vs South',
'city_pop': 'Population of the city',
'job_1': 'Occupation income level',
'job_2': 'Occupation education requirement',
'job_3': 'Occupation career level',
'job_4': 'Occupation cyber knowledge',
'cardholder_age': 'Cardholder age',
'loc_cluster_issue': 'Location Cluster Card Issue',
'loc_cluster_transac': 'Location Cluster Transaction',
'issue_transac_dist': 'Distance between where card was issued and transaction made',
# D10 client-level features (present only in fraudTrain_encoded_client.csv)
'card_amt_zscore': 'Card amount vs own history (z-score)',
'card_hours_since_prev': 'Card hours since previous transaction',
'card_txn_1h': 'Card transactions in past 1h',
'card_txn_24h': 'Card transactions in past 24h',
'card_txn_7d': 'Card transactions in past 7d',
'card_dist_from_usual_km': 'Card distance from usual location (km)',
'card_new_category': 'Card first purchase in category',
}
# V26: readable labels for the raw-baseline columns (RAW_BASELINE), so its explanations are read
# on the same terms as the encoded model's: category names, units, plain descriptions.
# 'amt', 'city_pop' and 'gender' already have labels above.
_CATEGORY_NAMES = {'entertainment': 'entertainment', 'food_dining': 'food and dining',
    'gas_transport': 'gas and transport', 'grocery_net': 'grocery (online)',
    'grocery_pos': 'grocery (in store)', 'health_fitness': 'health and fitness',
    'home': 'home products', 'kids_pets': 'kids and pets', 'misc_net': 'miscellaneous (online)',
    'misc_pos': 'miscellaneous (in store)', 'personal_care': 'personal care',
    'shopping_net': 'shopping (online)', 'shopping_pos': 'shopping (in store)', 'travel': 'travel'}
column_names_dict.update({
    'lat': 'Cardholder home latitude', 'long': 'Cardholder home longitude',
    'merch_lat': 'Merchant latitude', 'merch_long': 'Merchant longitude',
    'age': 'Cardholder age', 'hour_of_day': 'Hour of day (0-23)',
    'day_of_week': 'Day of week (0 = Monday)',
    'city_ordinal': 'City (alphabetical code)', 'job_ordinal': 'Occupation (alphabetical code)',
    **{f'category_{k}': f'Category = {v}' for k, v in _CATEGORY_NAMES.items()},
})
if RAW_BASELINE:
    column_names_dict.update({c: f"State = {c[len('state_'):]}" for c in df_enc.columns
                              if c.startswith('state_')})
    # the raw baseline must not reuse an encoded-only label for a different quantity
    for _c in ('category_1', 'category_2', 'category_3', 'category_4a', 'category_4b', 'category_4c',
               'category_interaction_1', 'category_interaction_2', 'cardholder_age'):
        assert _c not in df_enc.columns



In [ ]:
df_enc = df_enc.rename(columns=column_names_dict)

df_enc

In [ ]:
print("Credit Card Fraud Detection data -  rows:",df_enc.shape[0]," columns:", df_enc.shape[1])

#### Set Train and Test features and data

In [ ]:
target = 'is_fraud'
# V26: predictors are the data file's columns in file order, renamed -- the order every Phase 7
# script uses (revision_scripts/ablation_run.py). For the encoded files this is exactly V25's
# hardcoded list (asserted below), so nothing changes for them.
_V25_ENCODED = ['Time difference from the first transaction', 'Transaction time group',
       'Offline vs. online', 'Non-essential vs. essential',
       'Average item cost', 'Purpose of purchase: household/wellbeing',
       'Purpose of purchase: entertainment/lifestyle',
       'Purpose of purchase: shopping-related',
       'Interaction 1: Non-essential vs. essential*Average item cost',
       'Interaction 2: Non-essential vs. essential*Purpose of purchase: household or wellbeing',
       'Amount spent (USD)', 'Gender', 'City socioeconomic status',
       'City social inequality level', 'City Rural-Urban',
       'City high-speed fixed broadband coverage',
       'City high-speed mobile broadband', 'State West vs East',
       'State North vs South', 'Population of the city',
       'Occupation income level', 'Occupation education requirement',
       'Occupation career level', 'Occupation cyber knowledge',
       'Cardholder age', 'Location Cluster Card Issue',
       'Location Cluster Transaction',
       'Distance between where card was issued and transaction made']
predictors = [c for c in df_enc.columns if c != target]
assert all(c in column_names_dict.values() for c in predictors), "a column has no readable label"
if not RAW_BASELINE:
    assert predictors[:len(_V25_ENCODED)] == _V25_ENCODED


In [ ]:
# B4: see DROP_TRANS_TIME in cell 0. The column stays in df_enc either way,
# because the chronological split is cut on it; this only stops the models
# training on it.
if DROP_TRANS_TIME:
    predictors = [p for p in predictors if p != 'Time difference from the first transaction']
# D10: the seven client-level features are already in the file, after the other predictors.
CLIENT_PREDICTORS = ['Card amount vs own history (z-score)',
     'Card hours since previous transaction',
     'Card transactions in past 1h',
     'Card transactions in past 24h',
     'Card transactions in past 7d',
     'Card distance from usual location (km)',
     'Card first purchase in category']
assert all(c in predictors for c in CLIENT_PREDICTORS) == CLIENT_FEATURES
assert not CLIENT_FEATURES or predictors[-len(CLIENT_PREDICTORS):] == CLIENT_PREDICTORS
print(f"{len(predictors)} predictors | CONFIG={CONFIG} | TAG={TAG}")


In [ ]:
# D8 (Revision_Plan_V03.md): tuned hyperparameters, one set per data configuration.
# Each set comes from revision_scripts/d8_tuning.py -- a 40-draw random search on
# validation AUC-PR -- and a model's tuned set is kept only if it beat that model's
# untuned baseline on the same data; otherwise the baseline is listed here.
# Sources: result_csv/hyperparameter_tuning_V01.csv ("time"),
# result_csv/hyperparameter_tuning_V02_<config>.csv ("notime", "notime_client": pre-fix data, V25) and
# result_csv/hyperparameter_tuning_V03_<config>.csv (the p7_* entries: Phase 7, corrected data).
TUNED = {
    "time": {   # trans_time kept -- the configuration model_V23/V24 ran
        'Logistic Regression': {'C': 0.002550033665615218, 'class_weight': 'balanced', 'solver': 'liblinear'},
        'Decision Tree': {'max_depth': 10, 'min_samples_split': 50, 'min_samples_leaf': 20, 'criterion': 'gini', 'class_weight': None},
        'Random Forest': {'n_estimators': 400, 'max_depth': None, 'min_samples_split': 10, 'min_samples_leaf': 5, 'max_features': 0.5, 'class_weight': 'balanced_subsample'},
        'AdaBoost': {'n_estimators': 100, 'learning_rate': 0.8},   # untuned: search did not beat it
        'CatBoost': {'iterations': 1000, 'depth': 8, 'learning_rate': 0.010516599815699672, 'l2_leaf_reg': 1.0, 'bagging_temperature': 0.9116000718508169},
        'XGBoost': {'eta': 0.021295227344361605, 'max_depth': 8, 'subsample': 0.6845407914876798, 'colsample_bytree': 0.9685434809022035, 'min_child_weight': 1.0, 'gamma': 0.0, 'scale_pos_weight': 10.0},
        'LightGBM': {'learning_rate': 0.03291491138999078, 'num_leaves': 63, 'max_depth': 12, 'min_child_samples': 20, 'feature_fraction': 0.8963987700651181, 'bagging_fraction': 0.9109168770479202, 'scale_pos_weight': 1.0},
        'K-NN': {'n_neighbors': 50, 'weights': 'distance', 'p': 1},
    },
    "notime": {
        'Logistic Regression': {'C': 0.002550033665615218, 'class_weight': 'balanced', 'solver': 'liblinear'},
        'Decision Tree': {'max_depth': 14, 'min_samples_split': 2, 'min_samples_leaf': 100, 'criterion': 'entropy', 'class_weight': 'balanced'},
        'Random Forest': {'n_estimators': 400, 'max_depth': 16, 'min_samples_split': 2, 'min_samples_leaf': 5, 'max_features': 0.5, 'class_weight': None},
        'AdaBoost': {'n_estimators': 100, 'learning_rate': 0.8},   # untuned: search did not beat it
        'CatBoost': {'iterations': 1000, 'depth': 8, 'learning_rate': 0.015474590024779264, 'l2_leaf_reg': 1.0, 'bagging_temperature': 0.12479937612392289},
        'XGBoost': {'eta': 0.021295227344361605, 'max_depth': 8, 'subsample': 0.6845407914876798, 'colsample_bytree': 0.9685434809022035, 'min_child_weight': 1.0, 'gamma': 0.0, 'scale_pos_weight': 10.0},
        'LightGBM': {'learning_rate': 0.03299276902809232, 'num_leaves': 127, 'max_depth': 12, 'min_child_samples': 50, 'feature_fraction': 0.7374854068785813, 'bagging_fraction': 0.8553126544245916, 'scale_pos_weight': 1.0},
        'K-NN': {'n_neighbors': 50, 'weights': 'distance', 'p': 2},
    },
    "notime_client": {
        'Logistic Regression': {'C': 36.861853832288666, 'class_weight': None, 'solver': 'liblinear'},
        'Decision Tree': {'max_depth': 14, 'min_samples_split': 200, 'min_samples_leaf': 20, 'criterion': 'entropy', 'class_weight': None},
        'Random Forest': {'n_estimators': 400, 'max_depth': None, 'min_samples_split': 10, 'min_samples_leaf': 5, 'max_features': 0.5, 'class_weight': 'balanced_subsample'},
        'AdaBoost': {'n_estimators': 200, 'learning_rate': 0.9015315884130458},
        'CatBoost': {'iterations': 1000, 'depth': 8, 'learning_rate': 0.02376362784255082, 'l2_leaf_reg': 1.0, 'bagging_temperature': 0.8132572916585687},
        'XGBoost': {'eta': 0.027355308155894122, 'max_depth': 10, 'subsample': 0.9653476109156784, 'colsample_bytree': 0.895210791225691, 'min_child_weight': 1.0, 'gamma': 5.0, 'scale_pos_weight': 10.0},
        'LightGBM': {'learning_rate': 0.03299276902809232, 'num_leaves': 127, 'max_depth': 12, 'min_child_samples': 50, 'feature_fraction': 0.7374854068785813, 'bagging_fraction': 0.8553126544245916, 'scale_pos_weight': 1.0},
        'K-NN': {'n_neighbors': 50, 'weights': 'distance', 'p': 2},
    },
    "p7_notime": {
        'Logistic Regression': {'C': 0.002550033665615218, 'class_weight': 'balanced', 'solver': 'liblinear'},
        'Decision Tree': {'max_depth': 14, 'min_samples_split': 2, 'min_samples_leaf': 100, 'criterion': 'entropy', 'class_weight': 'balanced'},
        'Random Forest': {'n_estimators': 400, 'max_depth': 16, 'min_samples_split': 2, 'min_samples_leaf': 5, 'max_features': 0.5, 'class_weight': None},
        'AdaBoost': {'n_estimators': 200, 'learning_rate': 0.3735118623348581},
        'CatBoost': {'iterations': 1000, 'depth': 8, 'learning_rate': 0.010516599815699672, 'l2_leaf_reg': 1.0, 'bagging_temperature': 0.9116000718508169},
        'XGBoost': {'eta': 0.021295227344361605, 'max_depth': 8, 'subsample': 0.6845407914876798, 'colsample_bytree': 0.9685434809022035, 'min_child_weight': 1.0, 'gamma': 0.0, 'scale_pos_weight': 10.0},
        'LightGBM': {'learning_rate': 0.03291491138999078, 'num_leaves': 63, 'max_depth': 12, 'min_child_samples': 20, 'feature_fraction': 0.8963987700651181, 'bagging_fraction': 0.9109168770479202, 'scale_pos_weight': 1.0},
        'K-NN': {'n_neighbors': 50, 'weights': 'distance', 'p': 2},
    },
    "p7_notime_client": {
        'Logistic Regression': {'C': 36.861853832288666, 'class_weight': None, 'solver': 'liblinear'},
        'Decision Tree': {'max_depth': 14, 'min_samples_split': 200, 'min_samples_leaf': 20, 'criterion': 'entropy', 'class_weight': None},
        'Random Forest': {'n_estimators': 400, 'max_depth': 16, 'min_samples_split': 2, 'min_samples_leaf': 5, 'max_features': 0.5, 'class_weight': None},
        'AdaBoost': {'n_estimators': 400, 'learning_rate': 0.8894710233994896},
        'CatBoost': {'iterations': 1000, 'depth': 8, 'learning_rate': 0.015474590024779264, 'l2_leaf_reg': 1.0, 'bagging_temperature': 0.12479937612392289},
        'XGBoost': {'eta': 0.027355308155894122, 'max_depth': 10, 'subsample': 0.9653476109156784, 'colsample_bytree': 0.895210791225691, 'min_child_weight': 1.0, 'gamma': 5.0, 'scale_pos_weight': 10.0},
        'LightGBM': {'learning_rate': 0.03299276902809232, 'num_leaves': 127, 'max_depth': 12, 'min_child_samples': 50, 'feature_fraction': 0.7374854068785813, 'bagging_fraction': 0.8553126544245916, 'scale_pos_weight': 1.0},
        'K-NN': {'n_neighbors': 50, 'weights': 'distance', 'p': 2},
    },
    "p7_notime_raw": {
        'Logistic Regression': {'C': 73.45805338899959, 'class_weight': None, 'solver': 'liblinear'},
        'Decision Tree': {'max_depth': 10, 'min_samples_split': 50, 'min_samples_leaf': 20, 'criterion': 'gini', 'class_weight': None},
        'Random Forest': {'n_estimators': 400, 'max_depth': 16, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 0.5, 'class_weight': None},
        'AdaBoost': {'n_estimators': 200, 'learning_rate': 0.8198461683333294},
        'CatBoost': {'iterations': 1000, 'depth': 8, 'learning_rate': 0.015474590024779264, 'l2_leaf_reg': 1.0, 'bagging_temperature': 0.12479937612392289},
        'XGBoost': {'eta': 0.027355308155894122, 'max_depth': 10, 'subsample': 0.9653476109156784, 'colsample_bytree': 0.895210791225691, 'min_child_weight': 1.0, 'gamma': 5.0, 'scale_pos_weight': 10.0},
        'LightGBM': {'learning_rate': 0.03291491138999078, 'num_leaves': 63, 'max_depth': 12, 'min_child_samples': 20, 'feature_fraction': 0.8963987700651181, 'bagging_fraction': 0.9109168770479202, 'scale_pos_weight': 1.0},
        'K-NN': {'n_neighbors': 50, 'weights': 'distance', 'p': 2},
    },
    "p7_notime_raw_client": {
        'Logistic Regression': {'C': 11.210195684112005, 'class_weight': None, 'solver': 'lbfgs'},
        'Decision Tree': {'max_depth': 10, 'min_samples_split': 50, 'min_samples_leaf': 20, 'criterion': 'gini', 'class_weight': None},
        'Random Forest': {'n_estimators': 400, 'max_depth': None, 'min_samples_split': 10, 'min_samples_leaf': 5, 'max_features': 0.5, 'class_weight': 'balanced_subsample'},
        'AdaBoost': {'n_estimators': 400, 'learning_rate': 1.4301127567418113},
        'CatBoost': {'iterations': 1000, 'depth': 8, 'learning_rate': 0.02376362784255082, 'l2_leaf_reg': 1.0, 'bagging_temperature': 0.8132572916585687},
        'XGBoost': {'eta': 0.027355308155894122, 'max_depth': 10, 'subsample': 0.9653476109156784, 'colsample_bytree': 0.895210791225691, 'min_child_weight': 1.0, 'gamma': 5.0, 'scale_pos_weight': 10.0},
        'LightGBM': {'learning_rate': 0.03299276902809232, 'num_leaves': 127, 'max_depth': 12, 'min_child_samples': 50, 'feature_fraction': 0.7374854068785813, 'bagging_fraction': 0.8553126544245916, 'scale_pos_weight': 1.0},
        'K-NN': {'n_neighbors': 50, 'weights': 'distance', 'p': 2},
    },
    # "p7_*" entries are filled in by revision_scripts/fill_tuned.py --version V03 once their searches finish
}
assert CONFIG in TUNED, f"no tuned hyperparameters for CONFIG={CONFIG!r} yet"
PARAMS = TUNED[CONFIG]
pd.DataFrame({m: [str(p)] for m, p in PARAMS.items()}, index=["hyperparameters"]).T


In [ ]:
# D6: chronological split. trans_time is hours since the first transaction, a
# monotone transform of unix_time, so cutting on it is equivalent to cutting on
# the timestamp; the cut is on the value rather than the row index, so no hour
# straddles a boundary. The submitted design shuffled rows, which put the same
# 983 cardholders on both sides of every boundary.
def make_split(mode=None):
    mode = mode or SPLIT_MODE
    if mode == "random":
        tr, te = train_test_split(df_enc, test_size=TEST_SIZE,
                                  random_state=RANDOM_STATE, shuffle=True)
        tr, va = train_test_split(tr, test_size=VALID_SIZE,
                                  random_state=RANDOM_STATE, shuffle=True)
        return tr, va, te
    if mode == "temporal":
        t = df_enc["Time difference from the first transaction"]
        c1, c2 = t.quantile(0.64), t.quantile(0.80)
        return df_enc[t <= c1], df_enc[(t > c1) & (t <= c2)], df_enc[t > c2]
    raise ValueError(f"unknown SPLIT_MODE: {mode}")


train_df, valid_df, test_df = make_split()
print(f"{SPLIT_MODE} split -> train {len(train_df)}, valid {len(valid_df)}, test {len(test_df)}")
print("fraud:", int(train_df[target].sum()), int(valid_df[target].sum()), int(test_df[target].sum()))


In [ ]:
train_df[target].value_counts()

In [ ]:
train_df

In [ ]:
valid_df[target].value_counts()

#### Handling of Imbalance

In [ ]:
# D7 (Revision_Plan_V02.md; re-run without trans_time, Revision_Plan_V04.md): NO RESAMPLING.
#
# The sweep (25 configurations, default hyperparameters, validation AUC-PR, chronological
# split, trans_time dropped) puts no resampling 5th on the mean across models (0.5625)
# but within 0.003 of the best setting (ROS 10:90, 0.5657). Per model, default LightGBM
# gains 0.057 from ROS 10:90, every other model 0.018 or less
# (result_csv/best_sampling_per_model_V01_notime.csv).
#
# With each model's TUNED hyperparameters (revision_scripts/tuned_resampling_check.py)
# LightGBM's gain falls to about 0.009 and the other models gain nothing or lose, so
# no model is resampled.
#
# Why: AUC-PR measures ranking, and resampling mainly shifts the predicted
# probabilities (the class prior the model sees) rather than their ordering. That holds
# whether or not a model weights the minority class in its loss -- five of the eight do;
# CatBoost, XGBoost and k-NN do not unless tuning selects it.
#
# Set RESAMPLING to "ROS" to reproduce the submitted configuration.
RESAMPLING = None   # None | "ROS" | "RUS" | "SMOTE"
SAMPLING_STRATEGY = 0.0066667   # 1:150, used only when RESAMPLING is not None

if RESAMPLING is not None:
    sampler = {"ROS": RandomOverSampler, "RUS": RandomUnderSampler,
               "SMOTE": SMOTE}[RESAMPLING](sampling_strategy=SAMPLING_STRATEGY,
                                           random_state=RANDOM_STATE)
    X, y = sampler.fit_resample(train_df[predictors], train_df[target])
    train_df = pd.DataFrame(X, columns=predictors)
    train_df[target] = y
    print(f"{RESAMPLING} {SAMPLING_STRATEGY}: train is now {len(train_df)} rows")
else:
    print(f"no resampling: train stays at {len(train_df)} rows")

train_df[target].value_counts()


In [ ]:
# (handled in the cell above)
train_df


In [ ]:
train_df[target].value_counts()

#### Create Result DataFrame

In [ ]:
result_df = pd.DataFrame({'metric\model':['Recall', 'Precision', 'Specificity', 'F1 Score', 'AUC', 'AUC-PR', 'MCC', 'Balanced Classification Rate']})

In [ ]:
# D9 (Revision_Plan_V02.md): budget-based operating point.
#
# The submitted paper thresholded every model at 0.5 without justification.
# A card issuer does not choose a probability, it chooses how many alerts its
# investigators can work. ALERT_RATE fixes that budget and the threshold falls
# out of it, which is both defensible and directly comparable across models.
#
# B5: the rule cannot always be satisfied. An unpruned decision tree and a
# 1-nearest-neighbour classifier emit probabilities of only 0 or 1, so once
# fewer than ALERT_RATE of rows score 1 the k-th largest score is 0 and the
# rule would alert every transaction. Detect that and report "no operating
# point exists" rather than a spurious recall of 1.0 at base-rate precision.
def budget_threshold(preds_proba, alert_rate=ALERT_RATE):
    """Return (threshold, achieved_alert_rate, usable)."""
    p = np.asarray(preds_proba, dtype=float)
    k = max(1, int(round(len(p) * alert_rate)))
    thr = float(np.partition(p, -k)[-k])
    achieved = float((p >= thr).mean())
    usable = achieved <= alert_rate * 2   # tolerate ties up to twice the budget
    return thr, achieved, usable


def score_at_budget(model_name, y_true, preds_proba):
    """Record the operating point for one model into budget_df."""
    y_true = np.asarray(y_true)
    thr, achieved, usable = budget_threshold(preds_proba)
    if not usable:
        print(f"{model_name}: scores too coarse for a {ALERT_RATE:.1%} operating "
              f"point (threshold {thr:g} would alert {achieved:.1%} of transactions)")
        budget_df[model_name] = [np.nan] * 6
        return
    alerted = (np.asarray(preds_proba) >= thr).astype(int)
    row = [thr, achieved,
           recall_score(y_true, alerted),
           precision_score(y_true, alerted, zero_division=0),
           f1_score(y_true, alerted),
           matthews_corrcoef(y_true, alerted)]
    print(f"{model_name}: threshold {thr:.4f}, alert rate {achieved:.2%}, "
          f"recall {row[2]:.4f}, precision {row[3]:.4f}")
    budget_df[model_name] = row


budget_df = pd.DataFrame({'metric\\model': [
    'Operating threshold', 'Alert rate', 'Recall (budget)',
    'Precision (budget)', 'F1 (budget)', 'MCC (budget)']})


# V24: every fitted model is registered here, together with the operating
# threshold set on the validation set, so the test set can be scored once, at
# the end, with nothing re-fitted and nothing re-tuned. The notebook reuses the
# names `clf` and `model` for every estimator, so each has to be captured at the
# point it is still live -- the cell straight after its budget cell.
fitted_models = {}      # model_name -> function(dataframe) -> fraud probability
valid_thresholds = {}   # model_name -> (threshold, achieved_rate, usable) on valid_df


In [ ]:
feature_df = pd.DataFrame({'feature\model':predictors})

#### Logistic Regression

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(train_df[predictors])
X_test = scaler.transform(valid_df[predictors])

In [ ]:
# D8: hyperparameters from the TUNED table (after the predictor list).
model = LogisticRegression(max_iter=1000, **PARAMS["Logistic Regression"])
model_name = "Logistic Regression"


In [ ]:
model.fit(X_train, train_df[target])

In [ ]:
preds = model.predict(X_test)
preds_proba = model.predict_proba(X_test)[:, 1]

##### feature importance

In [ ]:
tmp = feature_importances = pd.DataFrame(model.coef_.flatten(), index=predictors, columns=['Feature importance'])
tmp = tmp.sort_values(by='Feature importance',ascending=False)

plt.figure(figsize = (7,4))
plt.title('Features importance',fontsize=14)
s = sns.barplot(x=tmp.index,y='Feature importance',data=tmp)
s.set_xticklabels(s.get_xticklabels(),rotation=90)
plt.show()

##### confusion matrix

In [ ]:
cm = pd.crosstab(valid_df[target].values, preds, rownames=['Actual'], colnames=['Predicted'])
display(HTML(cm.to_html(index=True,header=True)))

fig, (ax1) = plt.subplots(ncols=1, figsize=(5,5))
sns.heatmap(cm, 
            xticklabels=['Not Fraud', 'Fraud'],
            yticklabels=['Not Fraud', 'Fraud'],
            annot=True, ax=ax1,
            linewidths=.2, linecolor="Darkblue", cmap="Blues")

##### score

In [ ]:
recall = recall_score(valid_df[target].values, preds, average='binary')
print(f"Recall Score of {model_name} = ", recall)

precision = precision_score(valid_df[target].values, preds, average='binary')
print(f"Precision Score of {model_name} = ", precision)

specificity = recall_score(valid_df[target].values, preds, pos_label=0)
print(f"Specificity Score of {model_name} = ", specificity)

f1 = f1_score(valid_df[target].values, preds)
print(f"F1 Score of {model_name} = ", f1)

In [ ]:
fpr, tpr, threshold = metrics.roc_curve(valid_df[target].values, preds_proba)
roc_auc = metrics.auc(fpr, tpr)

plt.title(f"Receiver Operating Characteristic for {model_name}")
plt.plot(fpr, tpr, 'r', label = 'AUC = %0.2f' % roc_auc)
plt.legend(loc = 'lower right')
plt.plot([0, 1], [0, 1],'g--')
plt.xlim([0, 1])
plt.ylim([0, 1])
plt.ylabel('True Positive Rate')
plt.xlabel('False Positive Rate')
plt.show()

In [ ]:
AUC = roc_auc_score(valid_df[target], preds_proba)
AUCPR = average_precision_score(valid_df[target], preds_proba)

print(f"AUC of {model_name} = ", AUC)
print(f"AUC-PR of {model_name} = ", AUCPR)

AUC, AUCPR


In [ ]:
# D9: operating point at the ALERT_RATE investigation budget
score_at_budget(model_name, valid_df[target].values, preds_proba)

budget_df


In [ ]:
# V24: register Logistic Regression for the one-time test evaluation
assert model_name == "Logistic Regression", model_name
fitted_models[model_name] = lambda d, m=model, s=scaler: m.predict_proba(s.transform(d[predictors]))[:, 1]
valid_thresholds[model_name] = budget_threshold(preds_proba)


In [ ]:
MCC = matthews_corrcoef(valid_df[target].values, preds)
print(f"Matthew's Correlation Coefficient of {model_name} = ", MCC)

bcr = (recall+specificity)/2
print(f"Balanced Classification Rate of {model_name} = ", bcr)

In [ ]:
result_df[model_name] = [recall, precision, specificity, f1, AUC, AUCPR, MCC, bcr]

result_df

#### Decision Tree

In [ ]:
# D8: hyperparameters from the TUNED table (after the predictor list).
# The submitted tree was grown unpruned; tuned, it is the interpretability
# result of the revision (Revision_Phase2_Results_V03.md).
clf = DecisionTreeClassifier(**PARAMS["Decision Tree"], random_state=RANDOM_STATE)
model_name = "Decision Tree"


In [ ]:
clf.fit(train_df[predictors], train_df[target].values)

In [ ]:
train_df

In [ ]:
preds = clf.predict(valid_df[predictors])
preds_proba = clf.predict_proba(valid_df[predictors])[:, 1] # Phase 1: needed for a real ROC-AUC and AUC-PR

##### tree plot

In [ ]:
plt.figure(figsize=(200, 30))

plot_tree(clf, 
filled=True,
max_depth=6,
label='root',
feature_names = predictors,
class_names = ["Not fraud", "Fraud"],
rounded=True,
impurity=False,
fontsize=10
)

plt.show()

##### feature importance

In [ ]:
tmp = pd.DataFrame({'Feature': predictors, 'Feature importance': clf.feature_importances_})
tmp = tmp.sort_values(by='Feature importance',ascending=False)
plt.figure(figsize = (7,4))
plt.title('Features importance',fontsize=14)
s = sns.barplot(x='Feature',y='Feature importance',data=tmp)
s.set_xticklabels(s.get_xticklabels(),rotation=90)
plt.show()

In [ ]:
feature_imp = pd.DataFrame({'Feature': predictors, 'Feature importance': clf.feature_importances_})

feature_imp

##### SHAP

In [ ]:
# Obtain random entries and get SHAP value

nonFraud_index = list(valid_df.loc[valid_df[target]==0].index) # obtain non Fraud index
random.seed(RANDOM_STATE) # set random state number
random_nonFraud = np.array(random.sample(nonFraud_index, 1000)) # obtain indeces from non Fraud

Fraud_index = list(valid_df.loc[valid_df[target]==1].index) # obtain Fraud index
random.seed(RANDOM_STATE) # set random state number
random_Fraud = np.array(random.sample(Fraud_index, 1000)) # obtain indeces from Fraud

random_index = np.concatenate((random_nonFraud, random_Fraud), axis=0)
random.shuffle(random_index)

df_random = valid_df.loc[random_index,:] # df of randomly selected data

df_random['is_fraud'].value_counts()

In [ ]:
# SHAP beeswarm for random samples (class ratio is aprox. same with original)

explainer = shap.TreeExplainer(clf)
X = df_random[predictors] # obtain selected random entries for shap

shap_values = explainer(X)

print(shap_values.shape)

exp = shap.Explanation(shap_values.values[:,:,1], 
                  shap_values.base_values[:,1], 
                  data=X.values, 
                  feature_names=X.columns)
idx = 0

print(f"SHAP beeswarm, both Non fraud and fraud, {model_name}:\n")

for f in predictors:
    data = df_random[f]
    print(f"{f}: \t minium: {data.min()}, median: {data.median()}, max: {data.max()}")
    
shap.plots.beeswarm(exp, max_display=20, color=plt.get_cmap("rainbow"))

In [ ]:
# SHAP force graph

shap.initjs()
shap.plots.force(explainer.expected_value[0], exp[:].values, feature_names=predictors)

##### LIME

In [ ]:
print(f"LIME, {model_name}:")

start = datetime.now()
print("start:", start)

warnings.filterwarnings('ignore') # disable Python warning for this cell

# V25: seeded -- without random_state LIME draws different perturbation samples each
# run, so the averaged contributions changed between otherwise identical runs
# (by up to 0.0045; the top-ranked conditions were stable).
explainer = lime.lime_tabular.LimeTabularExplainer(train_df[predictors].values,
random_state=RANDOM_STATE,
feature_names=predictors,
class_names=['class_0', 'class_1'],
discretize_continuous=True)

print("Baseline probability:", valid_df[target].values.mean()) # obtain baseline probability

n = 500
contributions_list = []
for i in range(n):
    exp = explainer.explain_instance(df_random[predictors].iloc[i], clf.predict_proba, num_features=len(predictors), num_samples=2000)
    contributions = exp.as_list()  # [(feature1, value1), (feature2, value2), ...]
    contributions_list.extend(contributions)

contributions_dict = {}
for feature, value in contributions_list:
    if feature not in contributions_dict:
        contributions_dict[feature] = []
    contributions_dict[feature].append(value)

for k, v in contributions_dict.items():
    average = np.mean(v)
    contributions_dict[k] = average

df_lime = pd.DataFrame.from_dict(contributions_dict, orient='index')

df_lime.to_csv(f"result_csv/LIME_{model_name}_{TAG}.csv")

end = datetime.now()
print("end:", end)
print("time required:", end-start)

df_lime

##### confusion matrix

In [ ]:
cm = pd.crosstab(valid_df[target].values, preds, rownames=['Actual'], colnames=['Predicted'])
display(HTML(cm.to_html(index=True,header=True)))

fig, (ax1) = plt.subplots(ncols=1, figsize=(5,5))
sns.heatmap(cm, 
            xticklabels=['Not Fraud', 'Fraud'],
            yticklabels=['Not Fraud', 'Fraud'],
            annot=True, ax=ax1,
            linewidths=.2, linecolor="Darkblue", cmap="Blues")

##### score 

In [ ]:
recall = recall_score(valid_df[target].values, preds, average='binary')
print(f"Recall Score of {model_name} = ", recall)

precision = precision_score(valid_df[target].values, preds, average='binary')
print(f"Precision Score of {model_name} = ", precision)

specificity = recall_score(valid_df[target].values, preds, pos_label=0)
print(f"Specificity Score of {model_name} = ", specificity)

f1 = f1_score(valid_df[target].values, preds)
print(f"F1 Score of {model_name} = ", f1)    

In [ ]:
fpr, tpr, threshold = metrics.roc_curve(valid_df[target].values, preds)
roc_auc = metrics.auc(fpr, tpr)

plt.title(f"Receiver Operating Characteristic for {model_name}")
plt.plot(fpr, tpr, 'r', label = 'AUC = %0.2f' % roc_auc)
plt.legend(loc = 'lower right')
plt.plot([0, 1], [0, 1],'g--')
plt.xlim([0, 1])
plt.ylim([0, 1])
plt.ylabel('True Positive Rate')
plt.xlabel('False Positive Rate')
plt.show()

In [ ]:
AUC = roc_auc_score(valid_df[target].values, preds_proba)
AUCPR = average_precision_score(valid_df[target].values, preds_proba)

print(f"AUC of {model_name} = ", AUC)
print(f"AUC-PR of {model_name} = ", AUCPR)

AUC, AUCPR


In [ ]:
# D9: operating point at the ALERT_RATE investigation budget
score_at_budget(model_name, valid_df[target].values, preds_proba)

budget_df


In [ ]:
# V24: register Decision Tree for the one-time test evaluation
assert model_name == "Decision Tree", model_name
fitted_models[model_name] = lambda d, m=clf: m.predict_proba(d[predictors])[:, 1]
valid_thresholds[model_name] = budget_threshold(preds_proba)


In [ ]:
MCC = matthews_corrcoef(valid_df[target].values, preds)
print(f"Matthew's Correlation Coefficient of {model_name} = ", MCC)

bcr = (recall+specificity)/2
print(f"Balanced Classification Rate of {model_name} = ", bcr)

##### Result df

In [ ]:
result_df[model_name] = [recall, precision, specificity, f1, AUC, AUCPR, MCC, bcr]

result_df

In [ ]:
feature_df[model_name] = feature_imp['Feature importance']

feature_df

#### Random Forest

In [ ]:
# D8: hyperparameters from the TUNED table (after the predictor list).
clf = RandomForestClassifier(**PARAMS["Random Forest"], n_jobs=NO_JOBS,
                             random_state=RANDOM_STATE, verbose=False)
model_name = "Random Forest"


In [ ]:
clf.fit(train_df[predictors], train_df[target].values)

In [ ]:
preds = clf.predict(valid_df[predictors])
preds_proba = clf.predict_proba(valid_df[predictors])[:, 1] # Phase 1: needed for a real ROC-AUC and AUC-PR

##### feature importance

In [ ]:
tmp = pd.DataFrame({'Feature': predictors, 'Feature importance': clf.feature_importances_})
tmp = tmp.sort_values(by='Feature importance',ascending=False)
plt.figure(figsize = (7,4))
plt.title('Features importance',fontsize=14)
s = sns.barplot(x='Feature',y='Feature importance',data=tmp)
s.set_xticklabels(s.get_xticklabels(),rotation=90)
plt.show()

In [ ]:
feature_imp = pd.DataFrame({'Feature': predictors, 'Feature importance': clf.feature_importances_})

feature_imp

##### SHAP

In [ ]:
# SHAP beeswarm for random samples (class ratio is aprox. same with original)

explainer = shap.TreeExplainer(clf)
X = df_random[predictors] # obtain selected random entries for shap

shap_values = explainer(X)

print(shap_values.shape)

exp = shap.Explanation(shap_values.values[:,:,1], 
                  shap_values.base_values[:,1], 
                  data=X.values, 
                  feature_names=X.columns)
idx = 0

print(f"SHAP beeswarm, both Non fraud and fraud, {model_name}:\n")

for f in predictors:
    data = df_random[f]
    print(f"{f}: \t minium: {data.min()}, median: {data.median()}, max: {data.max()}")
    
shap.plots.beeswarm(exp, max_display=20, color=plt.get_cmap("rainbow"))

In [ ]:
# SHAP force graph

shap.initjs()
shap.plots.force(explainer.expected_value[0], exp[:].values, feature_names=predictors)

##### LIME

In [ ]:
print(f"LIME, {model_name}:")

start = datetime.now()
print("start:", start)

warnings.filterwarnings('ignore') # disable Python warning for this cell

# V25: seeded -- without random_state LIME draws different perturbation samples each
# run, so the averaged contributions changed between otherwise identical runs
# (by up to 0.0045; the top-ranked conditions were stable).
explainer = lime.lime_tabular.LimeTabularExplainer(train_df[predictors].values,
random_state=RANDOM_STATE,
feature_names=predictors,
class_names=['class_0', 'class_1'],
discretize_continuous=True)

print("Baseline probability:", valid_df[target].values.mean()) # obtain baseline probability

n = 500
contributions_list = []
for i in range(n):
    exp = explainer.explain_instance(df_random[predictors].iloc[i], clf.predict_proba, num_features=len(predictors), num_samples=2000)
    contributions = exp.as_list()  # [(feature1, value1), (feature2, value2), ...]
    contributions_list.extend(contributions)

contributions_dict = {}
for feature, value in contributions_list:
    if feature not in contributions_dict:
        contributions_dict[feature] = []
    contributions_dict[feature].append(value)

for k, v in contributions_dict.items():
    average = np.mean(v)
    contributions_dict[k] = average

df_lime = pd.DataFrame.from_dict(contributions_dict, orient='index')

df_lime.to_csv(f"result_csv/LIME_{model_name}_{TAG}.csv")

end = datetime.now()
print("end:", end)
print("time required:", end-start)

df_lime

##### confusion matrix

In [ ]:
cm = pd.crosstab(valid_df[target].values, preds, rownames=['Actual'], colnames=['Predicted'])
display(HTML(cm.to_html(index=True,header=True)))

fig, (ax1) = plt.subplots(ncols=1, figsize=(5,5))
sns.heatmap(cm, 
            xticklabels=['Not Fraud', 'Fraud'],
            yticklabels=['Not Fraud', 'Fraud'],
            annot=True, ax=ax1,
            linewidths=.2, linecolor="Darkblue", cmap="Blues")

##### score 

In [ ]:
recall = recall_score(valid_df[target].values, preds, average='binary')
print(f"Recall Score of {model_name} = ", recall)

precision = precision_score(valid_df[target].values, preds, average='binary')
print(f"Precision Score of {model_name} = ", precision)

specificity = recall_score(valid_df[target].values, preds, pos_label=0)
print(f"Specificity Score of {model_name} = ", specificity)

f1 = f1_score(valid_df[target].values, preds)
print(f"F1 Score of {model_name} = ", f1) 

In [ ]:
fpr, tpr, threshold = metrics.roc_curve(valid_df[target].values, preds)
roc_auc = metrics.auc(fpr, tpr)

plt.title(f"Receiver Operating Characteristic for {model_name}")
plt.plot(fpr, tpr, 'r', label = 'AUC = %0.2f' % roc_auc)
plt.legend(loc = 'lower right')
plt.plot([0, 1], [0, 1],'g--')
plt.xlim([0, 1])
plt.ylim([0, 1])
plt.ylabel('True Positive Rate')
plt.xlabel('False Positive Rate')
plt.show()

In [ ]:
AUC = roc_auc_score(valid_df[target].values, preds_proba)
AUCPR = average_precision_score(valid_df[target].values, preds_proba)

print(f"AUC of {model_name} = ", AUC)
print(f"AUC-PR of {model_name} = ", AUCPR)

AUC, AUCPR


In [ ]:
# D9: operating point at the ALERT_RATE investigation budget
score_at_budget(model_name, valid_df[target].values, preds_proba)

budget_df


In [ ]:
# V24: register Random Forest for the one-time test evaluation
assert model_name == "Random Forest", model_name
fitted_models[model_name] = lambda d, m=clf: m.predict_proba(d[predictors])[:, 1]
valid_thresholds[model_name] = budget_threshold(preds_proba)


In [ ]:
MCC = matthews_corrcoef(valid_df[target].values, preds)
print(f"Matthew's Correlation Coefficient of {model_name} = ", MCC)

bcr = (recall+specificity)/2
print(f"Balanced Classification Rate of {model_name} = ", bcr)

##### Result df

In [ ]:
result_df[model_name] = [recall, precision, specificity, f1, AUC, AUCPR, MCC, bcr]

result_df

In [ ]:
feature_df[model_name] = feature_imp['Feature importance']

feature_df

#### AdaBoost

In [ ]:
# D8: hyperparameters from the TUNED table (after the predictor list).
clf = AdaBoostClassifier(**PARAMS["AdaBoost"], algorithm='SAMME', random_state=RANDOM_STATE)

model_name = "AdaBoost"


In [ ]:
class_weights = compute_class_weight('balanced', classes=np.unique(train_df[target].values), y=train_df[target].values)
class_weight_dict = dict(enumerate(class_weights))

class_weight_dict

In [ ]:
sample_weight = np.array([class_weight_dict[class_label] for class_label in train_df[target].values])

In [ ]:
clf.fit(train_df[predictors], train_df[target].values, sample_weight=sample_weight)

In [ ]:
preds = clf.predict(valid_df[predictors])
preds_proba = clf.predict_proba(valid_df[predictors])[:, 1] # Phase 1: needed for a real ROC-AUC and AUC-PR

##### feature importance

In [ ]:
tmp = pd.DataFrame({'Feature': predictors, 'Feature importance': clf.feature_importances_})
tmp = tmp.sort_values(by='Feature importance',ascending=False)
plt.figure(figsize = (7,4))
plt.title('Features importance',fontsize=14)
s = sns.barplot(x='Feature',y='Feature importance',data=tmp)
s.set_xticklabels(s.get_xticklabels(),rotation=90)
plt.show()

##### confusion matrix

In [ ]:
cm = pd.crosstab(valid_df[target].values, preds, rownames=['Actual'], colnames=['Predicted'])
display(HTML(cm.to_html(index=True,header=True)))

fig, (ax1) = plt.subplots(ncols=1, figsize=(5,5))
sns.heatmap(cm, 
            xticklabels=['Not Fraud', 'Fraud'],
            yticklabels=['Not Fraud', 'Fraud'],
            annot=True,ax=ax1,
            linewidths=.2,linecolor="Darkblue", cmap="Blues")

##### score

In [ ]:
recall = recall_score(valid_df[target].values, preds, average='binary')
print(f"Recall Score of {model_name} = ", recall)

precision = precision_score(valid_df[target].values, preds, average='binary')
print(f"Precision Score of {model_name} = ", precision)

specificity = recall_score(valid_df[target].values, preds, pos_label=0)
print(f"Specificity Score of {model_name} = ", specificity)

f1 = f1_score(valid_df[target].values, preds)
print("F1 Score of AdaBoost = ", f1)

In [ ]:
fpr, tpr, threshold = metrics.roc_curve(valid_df[target].values, preds)
roc_auc = metrics.auc(fpr, tpr)

plt.figure(figsize = (5,4))
plt.title('Receiver Operating Characteristic for AdaBoost')
plt.plot(fpr, tpr, 'r', label = 'AUC = %0.2f' % roc_auc)
plt.legend(loc = 'lower right')
plt.plot([0, 1], [0, 1],'g--')
plt.xlim([0, 1])
plt.ylim([0, 1])
plt.ylabel('True Positive Rate')
plt.xlabel('False Positive Rate')
plt.show()

In [ ]:
AUC = roc_auc_score(valid_df[target].values, preds_proba)
AUCPR = average_precision_score(valid_df[target].values, preds_proba)

print(f"AUC of {model_name} = ", AUC)
print(f"AUC-PR of {model_name} = ", AUCPR)

AUC, AUCPR


In [ ]:
# D9: operating point at the ALERT_RATE investigation budget
score_at_budget(model_name, valid_df[target].values, preds_proba)

budget_df


In [ ]:
# V24: register AdaBoost for the one-time test evaluation
assert model_name == "AdaBoost", model_name
fitted_models[model_name] = lambda d, m=clf: m.predict_proba(d[predictors])[:, 1]
valid_thresholds[model_name] = budget_threshold(preds_proba)


In [ ]:
MCC = matthews_corrcoef(valid_df[target].values, preds)
print(f"Matthew's Correlation Coefficient of {model_name} = ", MCC)

bcr = (recall+specificity)/2
print(f"Balanced Classification Rate of {model_name} = ", bcr)

In [ ]:
result_df[model_name] = [recall, precision, specificity, f1, AUC, AUCPR, MCC, bcr]

result_df

#### Catboost

In [ ]:
# D8: hyperparameters from the TUNED table (after the predictor list).
clf = CatBoostClassifier(**PARAMS["CatBoost"], eval_metric='AUC', random_seed=RANDOM_STATE,
                         od_type='Iter', od_wait=100, metric_period=VERBOSE_EVAL)
model_name = "CatBoost"


In [ ]:
clf.fit(train_df[predictors], train_df[target].values,verbose=True)

In [ ]:
preds = clf.predict(valid_df[predictors])
preds_proba = clf.predict_proba(valid_df[predictors])[:, 1] # Phase 1: needed for a real ROC-AUC and AUC-PR

##### feature importance

In [ ]:
tmp = pd.DataFrame({'Feature': predictors, 'Feature importance': clf.feature_importances_/100})
tmp = tmp.sort_values(by='Feature importance',ascending=False)
plt.figure(figsize = (7,4))
plt.title('Features importance',fontsize=14)
s = sns.barplot(x='Feature',y='Feature importance',data=tmp)
s.set_xticklabels(s.get_xticklabels(),rotation=90)
plt.show()

In [ ]:
feature_imp = pd.DataFrame({'Feature': predictors, 'Feature importance': clf.feature_importances_})

feature_imp['Feature importance'] = feature_imp['Feature importance'] / 100

feature_imp

##### SHAP

In [ ]:
# SHAP beeswarm for random samples (class ratio is aprox. same with original)

# explainer = shap.TreeExplainer(clf, model_output="probability")
explainer = shap.Explainer(clf, link=shap.links.logit)
X = df_random[predictors] # obtain selected random entries for shap

shap_values = explainer(X)

print(shap_values.shape)

print(f"SHAP beeswarm, both Non fraud and fraud, {model_name}:\n")

for f in predictors:
    data = df_random[f]
    print(f"{f}: \t minium: {data.min()}, median: {data.median()}, max: {data.max()}")
    
shap.plots.beeswarm(shap_values, max_display=20, color=plt.get_cmap("rainbow"))

In [ ]:
# SHAP force graph

shap.initjs()
shap.plots.force(shap_values, feature_names=predictors)

##### LIME

In [ ]:
print(f"LIME, {model_name}:")

start = datetime.now()
print("start:", start)

warnings.filterwarnings('ignore') # disable Python warning for this cell

# V25: seeded -- without random_state LIME draws different perturbation samples each
# run, so the averaged contributions changed between otherwise identical runs
# (by up to 0.0045; the top-ranked conditions were stable).
explainer = lime.lime_tabular.LimeTabularExplainer(train_df[predictors].values,
random_state=RANDOM_STATE,
feature_names=predictors,
class_names=['class_0', 'class_1'],
discretize_continuous=True)

print("Baseline probability:", valid_df[target].values.mean()) # obtain baseline probability

n = 500
contributions_list = []
for i in range(n):
    exp = explainer.explain_instance(df_random[predictors].iloc[i], clf.predict_proba, num_features=len(predictors), num_samples=2000)
    contributions = exp.as_list()  # [(feature1, value1), (feature2, value2), ...]
    contributions_list.extend(contributions)

contributions_dict = {}
for feature, value in contributions_list:
    if feature not in contributions_dict:
        contributions_dict[feature] = []
    contributions_dict[feature].append(value)

for k, v in contributions_dict.items():
    average = np.mean(v)
    contributions_dict[k] = average

df_lime = pd.DataFrame.from_dict(contributions_dict, orient='index')

df_lime.to_csv(f"result_csv/LIME_{model_name}_{TAG}.csv")

end = datetime.now()
print("end:", end)
print("time required:", end-start)

df_lime

##### confusion matrix

In [ ]:
cm = pd.crosstab(valid_df[target].values, preds, rownames=['Actual'], colnames=['Predicted'])
display(HTML(cm.to_html(index=True,header=True)))

fig, (ax1) = plt.subplots(ncols=1, figsize=(5,5))
sns.heatmap(cm, 
            xticklabels=['Not Fraud', 'Fraud'],
            yticklabels=['Not Fraud', 'Fraud'],
            annot=True,ax=ax1,
            linewidths=.2,linecolor="Darkblue", cmap="Blues")

##### score

In [ ]:
recall = recall_score(valid_df[target].values, preds, average='binary')
print(f"Recall Score of {model_name} = ", recall)

precision = precision_score(valid_df[target].values, preds, average='binary')
print(f"Precision Score of {model_name} = ", precision)

specificity = recall_score(valid_df[target].values, preds, pos_label=0)
print(f"Specificity Score of {model_name} = ", specificity)

f1 = f1_score(valid_df[target].values, preds)
print(f"F1 Score of {model_name} = ", f1)

In [ ]:
fpr, tpr, threshold = metrics.roc_curve(valid_df[target].values, preds)
roc_auc = metrics.auc(fpr, tpr)

plt.figure(figsize = (5,4))
plt.title('Receiver Operating Characteristic for CatBoost')
plt.plot(fpr, tpr, 'r', label = 'AUC = %0.2f' % roc_auc)
plt.legend(loc = 'lower right')
plt.plot([0, 1], [0, 1],'g--')
plt.xlim([0, 1])
plt.ylim([0, 1])
plt.ylabel('True Positive Rate')
plt.xlabel('False Positive Rate')
plt.show()

In [ ]:
AUC = roc_auc_score(valid_df[target].values, preds_proba)
AUCPR = average_precision_score(valid_df[target].values, preds_proba)

print(f"AUC of {model_name} = ", AUC)
print(f"AUC-PR of {model_name} = ", AUCPR)

AUC, AUCPR


In [ ]:
# D9: operating point at the ALERT_RATE investigation budget
score_at_budget(model_name, valid_df[target].values, preds_proba)

budget_df


In [ ]:
# V24: register CatBoost for the one-time test evaluation
assert model_name == "CatBoost", model_name
fitted_models[model_name] = lambda d, m=clf: m.predict_proba(d[predictors])[:, 1]
valid_thresholds[model_name] = budget_threshold(preds_proba)


In [ ]:
MCC = matthews_corrcoef(valid_df[target].values, preds)
print(f"Matthew's Correlation Coefficient of {model_name} = ", MCC)

bcr = (recall+specificity)/2
print(f"Balanced Classification Rate of {model_name} = ", bcr)

In [ ]:
result_df[model_name] = [recall, precision, specificity, f1, AUC, AUCPR, MCC, bcr]

result_df

In [ ]:
feature_df[model_name] = feature_imp['Feature importance']

feature_df

#### XGBoost

In [ ]:
dtrain = xgb.DMatrix(train_df[predictors], train_df[target].values)
dvalid = xgb.DMatrix(valid_df[predictors], valid_df[target].values)
dtest = xgb.DMatrix(test_df[predictors], test_df[target].values)

watchlist = [(dtrain, 'train'), (dvalid, 'valid')] # What to monitor (in this case, **train** and **valid**)

# D8: hyperparameters from the TUNED table (after the predictor list).
# PARAMS comes last so an untuned baseline can carry its own eval_metric.
params = {'objective': 'binary:logistic', 'eval_metric': 'aucpr', 'seed': RANDOM_STATE,
          **PARAMS["XGBoost"]}

model_name = "XGBoost"


In [ ]:
model = xgb.train(params, 
                dtrain, 
                MAX_ROUNDS, 
                watchlist, 
                early_stopping_rounds=EARLY_STOP, 
                maximize=True, 
                verbose_eval=VERBOSE_EVAL) # defind as 50, means show iteration details every 50 interations

In [ ]:
class XGBoostWrapper:
    def __init__(self, model):
        self.model = model

    def predict_proba(self, data):
        dmatrix = xgb.DMatrix(data, feature_names=predictors)
        # Predict probabilities
        pred_probs = self.model.predict(dmatrix)
        # Ensure output is 2D
        if len(pred_probs.shape) == 1:
            pred_probs = np.vstack([1 - pred_probs, pred_probs]).transpose()
        return pred_probs

In [ ]:
preds = model.predict(dvalid)

##### feature importance

In [ ]:
_, ax = plt.subplots(1, 1)
xgb.plot_importance(model, height=0.8, title=f"Features importance {model_name}", ax=ax, color="green") 
plt.show()

##### feature importance (Table 4)


In [ ]:
# Decision 2 (Revision_Plan_V03.md): XGBoost replaces k-NN in the set of
# interpreted models -- k-NN is now the weakest model, XGBoost the strongest.
# Importance is total gain, normalised to sum to 1, so it is on the same scale as
# the impurity-based importances of the tree models in Table 4.
assert model_name == "XGBoost", model_name
_gain = model.get_score(importance_type='total_gain')
feature_imp = pd.DataFrame({'Feature': predictors,
                            'Feature importance': [_gain.get(f, 0.0) for f in predictors]})
feature_imp['Feature importance'] = feature_imp['Feature importance'] / feature_imp['Feature importance'].sum()
feature_df[model_name] = feature_imp['Feature importance']

feature_imp.sort_values('Feature importance', ascending=False)


##### SHAP


In [ ]:
# SHAP beeswarm for random samples (same balanced 2x1000 sample as the other models)
explainer = shap.TreeExplainer(model)
X = df_random[predictors]

shap_values = explainer(X)

print(shap_values.shape)
print(f"SHAP beeswarm, both Non fraud and fraud, {model_name}:\n")

shap.plots.beeswarm(shap_values, max_display=20, color=plt.get_cmap("rainbow"))


In [ ]:
# SHAP force graph

shap.initjs()
shap.plots.force(shap_values, feature_names=predictors)


##### LIME


In [ ]:
print(f"LIME, {model_name}:")

start = datetime.now()
print("start:", start)

warnings.filterwarnings('ignore') # disable Python warning for this cell

# V25: seeded -- without random_state LIME draws different perturbation samples each
# run, so the averaged contributions changed between otherwise identical runs
# (by up to 0.0045; the top-ranked conditions were stable).
explainer = lime.lime_tabular.LimeTabularExplainer(train_df[predictors].values,
random_state=RANDOM_STATE,
feature_names=predictors,
class_names=['class_0', 'class_1'],
discretize_continuous=True)

print("Baseline probability:", valid_df[target].values.mean()) # obtain baseline probability

n = 500
contributions_list = []
for i in range(n):
    exp = explainer.explain_instance(df_random[predictors].iloc[i], XGBoostWrapper(model).predict_proba, num_features=len(predictors), num_samples=2000)
    contributions = exp.as_list()  # [(feature1, value1), (feature2, value2), ...]
    contributions_list.extend(contributions)

contributions_dict = {}
for feature, value in contributions_list:
    if feature not in contributions_dict:
        contributions_dict[feature] = []
    contributions_dict[feature].append(value)

for k, v in contributions_dict.items():
    average = np.mean(v)
    contributions_dict[k] = average

df_lime = pd.DataFrame.from_dict(contributions_dict, orient='index')

df_lime.to_csv(f"result_csv/LIME_{model_name}_{TAG}.csv")

end = datetime.now()
print("end:", end)
print("time required:", end-start)

df_lime

##### get optimal threshold

In [ ]:
thshld_list = np.arange(0, 0.6, 0.025)

for th in thshld_list: # test best threshold value of turning probability into binary
    binary_preds = [1 if prob > th else 0 for prob in preds]
    print(f"Threshold of {th}:")
    cm = pd.crosstab(valid_df[target].values, binary_preds, rownames=['Actual'], colnames=['Predicted'])
    display(HTML(cm.to_html(index=True,header=True)))
    
    f1 = f1_score(valid_df[target].values, binary_preds)
    print(f"F1 Score of {model_name} = ", f1)
    
    fpr, tpr, threshold = metrics.roc_curve(valid_df[target].values, binary_preds)
    roc_auc = metrics.auc(fpr, tpr)
    plt.figure(figsize = (5,4))
    plt.title('Receiver Operating Characteristic for XGBoost')
    plt.plot(fpr, tpr, 'r', label = 'AUC = %0.2f' % roc_auc)
    plt.legend(loc = 'lower right')
    plt.plot([0, 1], [0, 1], 'g--')
    plt.xlim([0, 1])
    plt.ylim([0, 1])
    plt.ylabel('True Positive Rate')
    plt.xlabel('False Positive Rate')
    plt.show()

    print("\n\n")

In [ ]:
binary_preds = [1 if prob > 0.5 else 0 for prob in preds] # 0.5 for comparability with the other models; a stated threshold rule is Phase 2 (A4)

##### confusion matrix

In [ ]:
cm = pd.crosstab(valid_df[target].values, binary_preds, rownames=['Actual'], colnames=['Predicted'])

# print(valid_df[target].values)
# print(binary_preds)

display(HTML(cm.to_html(index=True,header=True)))

fig, (ax1) = plt.subplots(ncols=1, figsize=(5,5))
sns.heatmap(cm, 
            xticklabels=['Not Fraud', 'Fraud'],
            yticklabels=['Not Fraud', 'Fraud'],
            annot=True,ax=ax1,
            linewidths=.2,linecolor="Darkblue", cmap="Blues")

##### score

In [ ]:
recall = recall_score(valid_df[target].values, binary_preds, average='binary')
print(f"Recall Score of {model_name} = ", recall)

precision = precision_score(valid_df[target].values, binary_preds, average='binary')
print(f"Precision Score of {model_name} = ", precision)

specificity = recall_score(valid_df[target].values, binary_preds, pos_label=0)
print(f"Specificity Score of {model_name} = ", specificity)

f1 = f1_score(valid_df[target].values, binary_preds)
print(f"F1 Score of {model_name} = ", f1)

In [ ]:
fpr, tpr, threshold = metrics.roc_curve(valid_df[target].values, preds) # use probability instead of binary
roc_auc = metrics.auc(fpr, tpr)

plt.figure(figsize = (5,4))
plt.title(f'Receiver Operating Characteristic for {model_name}')
plt.plot(fpr, tpr, 'r', label = 'AUC = %0.2f' % roc_auc)
plt.legend(loc = 'lower right')
plt.plot([0, 1], [0, 1],'g--')
plt.xlim([0, 1])
plt.ylim([0, 1])
plt.ylabel('True Positive Rate')
plt.xlabel('False Positive Rate')
plt.show()

In [ ]:
roc_auc_score(valid_df[target].values, preds) # probability

In [ ]:
AUC = roc_auc_score(valid_df[target].values, preds)
AUCPR = average_precision_score(valid_df[target].values, preds)

print(f"AUC of {model_name} = ", AUC)
print(f"AUC-PR of {model_name} = ", AUCPR)

AUC, AUCPR


In [ ]:
# D9: operating point at the ALERT_RATE investigation budget
score_at_budget(model_name, valid_df[target].values, preds)

budget_df


In [ ]:
# V24: register XGBoost for the one-time test evaluation
assert model_name == "XGBoost", model_name
fitted_models[model_name] = lambda d, m=model: m.predict(xgb.DMatrix(d[predictors]))
valid_thresholds[model_name] = budget_threshold(preds)


In [ ]:
MCC = matthews_corrcoef(valid_df[target].values, binary_preds)
print(f"Matthew's Correlation Coefficient of {model_name} = ", MCC)

bcr = (recall+specificity)/2
print(f"Balanced Classification Rate of {model_name} = ", bcr)

In [ ]:
result_df[model_name] = [recall, precision, specificity, f1, AUC, AUCPR, MCC, bcr]

result_df

#### LightGBM

In [ ]:
# D8: hyperparameters from the TUNED table (after the predictor list).
params = {'boosting_type': 'gbdt', 'objective': 'binary', 'metric': 'average_precision',
          'bagging_freq': 1, 'nthread': 8, 'verbose': -1, 'seed': RANDOM_STATE,
          **PARAMS["LightGBM"]}

model_name = "LightGBM"


In [ ]:
# LightGBM rejects ':' and '*' in feature names, and the human-readable column
# names set earlier contain both (e.g. "Interaction 1: Non-essential vs.
# essential*Average item cost"). Passing them raises
#   [LightGBM] [Fatal] Do not support special JSON characters in feature name.
# In V22 that killed the cell, left `preds` holding the previous model's
# predictions, and the scoring cells below silently rescored XGBoost under
# LightGBM's name. Use positional names and keep the mapping for the plots.
lgb_feature_names = [f"f{i}" for i in range(len(predictors))]
lgb_feature_map = dict(zip(lgb_feature_names, predictors))

dtrain = lgb.Dataset(train_df[predictors].values,
                     label=train_df[target].values,
                     feature_name=lgb_feature_names)

dvalid = lgb.Dataset(valid_df[predictors].values,
                     label=valid_df[target].values,
                     feature_name=lgb_feature_names)


In [ ]:
evals_results = {}

model = lgb.train(params, 
                  dtrain, 
                  valid_sets=[dtrain, dvalid], 
                  valid_names=['train','valid'], 
                  # evals_result=evals_results, 
                  num_boost_round=MAX_ROUNDS,
                  # early_stopping_rounds=2*EARLY_STOP,
                  callbacks=[lgb.early_stopping(stopping_rounds=2*EARLY_STOP),],
                  # verbose_eval=VERBOSE_EVAL, 
                  feval=None)

In [ ]:
class LightGBMWrapper:
    def __init__(self, model):
        self.model = model

    def predict_proba(self, data):
        # Check if best_iteration exists, otherwise use the default
        if hasattr(self.model, 'best_iteration'):
            predictions = self.model.predict(data, num_iteration=self.model.best_iteration)
        else:
            predictions = self.model.predict(data)

        # Ensure that the prediction output is 2D (num_samples, num_classes)
        # If the predictions are for a binary classification problem
        if len(predictions.shape) == 1:
            predictions = np.vstack((1 - predictions, predictions)).T

        return predictions

In [ ]:
preds = model.predict(valid_df[predictors].values)

binary_preds = [1 if prob > 0.5 else 0 for prob in preds]

##### feature importance

In [ ]:
fig, (ax) = plt.subplots(ncols=1, figsize=(8,5))
lgb.plot_importance(model, height=0.8, title="Features importance (LightGBM)", ax=ax,color="red") 
plt.show()

##### confusion matrix

In [ ]:
cm = pd.crosstab(valid_df[target].values, binary_preds, rownames=['Actual'], colnames=['Predicted'])

# print(valid_df[target].values)
# print(binary_preds)

display(HTML(cm.to_html(index=True,header=True)))

fig, (ax1) = plt.subplots(ncols=1, figsize=(5,5))
sns.heatmap(cm, 
            xticklabels=['Not Fraud', 'Fraud'],
            yticklabels=['Not Fraud', 'Fraud'],
            annot=True,ax=ax1,
            linewidths=.2,linecolor="Darkblue", cmap="Blues")

##### score

In [ ]:
recall = recall_score(valid_df[target].values, binary_preds, average='binary')
print(f"Recall Score of {model_name} = ", recall)

precision = precision_score(valid_df[target].values, binary_preds, average='binary')
print(f"Precision Score of {model_name} = ", precision)

specificity = recall_score(valid_df[target].values, binary_preds, pos_label=0)
print(f"Specificity Score of {model_name} = ", specificity)

f1 = f1_score(valid_df[target].values, binary_preds)
print(f"F1 Score of {model_name} = ", f1)

In [ ]:
roc_auc_score(valid_df[target].values, preds) # probability, not binary

In [ ]:
fpr, tpr, threshold = metrics.roc_curve(valid_df[target].values, preds) # use probability instead of binary
roc_auc = metrics.auc(fpr, tpr)

plt.figure(figsize = (5,4))
plt.title(f'Receiver Operating Characteristic for {model_name}')
plt.plot(fpr, tpr, 'r', label = 'AUC = %0.2f' % roc_auc)
plt.legend(loc = 'lower right')
plt.plot([0, 1], [0, 1],'g--')
plt.xlim([0, 1])
plt.ylim([0, 1])
plt.ylabel('True Positive Rate')
plt.xlabel('False Positive Rate')
plt.show()

In [ ]:
AUC = roc_auc_score(valid_df[target].values, preds)
AUCPR = average_precision_score(valid_df[target].values, preds)

print(f"AUC of {model_name} = ", AUC)
print(f"AUC-PR of {model_name} = ", AUCPR)

AUC, AUCPR


In [ ]:
# D9: operating point at the ALERT_RATE investigation budget
score_at_budget(model_name, valid_df[target].values, preds)

budget_df


In [ ]:
# V24: register LightGBM for the one-time test evaluation
assert model_name == "LightGBM", model_name
fitted_models[model_name] = lambda d, m=model: m.predict(d[predictors].values)
valid_thresholds[model_name] = budget_threshold(preds)


In [ ]:
MCC = matthews_corrcoef(valid_df[target].values, binary_preds)
print(f"Matthew's Correlation Coefficient of {model_name} = ", MCC)

bcr = (recall+specificity)/2
print(f"Balanced Classification Rate of {model_name} = ", bcr)

In [ ]:
result_df[model_name] = [recall, precision, specificity, f1, AUC, AUCPR, MCC, bcr]

result_df

#### K-NN

In [ ]:
# D8: hyperparameters from the TUNED table (after the predictor list).
# k=1 was both the worst model and the one most inflated by the random split.
clf = KNeighborsClassifier(**PARAMS["K-NN"], n_jobs=-1)

model_name = "K-NN"


In [ ]:
clf.fit(train_df[predictors], train_df[target].values)

In [ ]:
preds = clf.predict_proba(valid_df[predictors])[:, 1]

binary_preds = [1 if prob > 0.5 else 0 for prob in preds]

##### LIME

*LIME is no longer run for k-NN.* Decision 2 in `Revision_Plan_V03.md`: k-NN is the weakest of the eight models (validation AUC-PR 0.150), so it was replaced by XGBoost in the set of interpreted models. k-NN stays in the performance comparison as a negative result.


##### confusion matrix

In [ ]:
cm = pd.crosstab(valid_df[target].values, binary_preds, rownames=['Actual'], colnames=['Predicted'])
display(HTML(cm.to_html(index=True,header=True)))

fig, (ax1) = plt.subplots(ncols=1, figsize=(5,5))
sns.heatmap(cm, 
            xticklabels=['Not Fraud', 'Fraud'],
            yticklabels=['Not Fraud', 'Fraud'],
            annot=True,ax=ax1,
            linewidths=.2,linecolor="Darkblue", cmap="Blues")

##### score

In [ ]:
recall = recall_score(valid_df[target].values, binary_preds, average='binary')
print(f"Recall Score of {model_name} = ", recall)

precision = precision_score(valid_df[target].values, binary_preds, average='binary')
print(f"Precision Score of {model_name} = ", precision)

specificity = recall_score(valid_df[target].values, binary_preds, pos_label=0)
print(f"Specificity Score of {model_name} = ", specificity)

f1 = f1_score(valid_df[target].values, binary_preds)
print(f"F1 Score of {model_name} = ", f1)

In [ ]:
fpr, tpr, threshold = metrics.roc_curve(valid_df[target].values, preds)
roc_auc = metrics.auc(fpr, tpr)

plt.figure(figsize = (5,4))
plt.title(f'Receiver Operating Characteristic for {model_name}')
plt.plot(fpr, tpr, 'r', label = 'AUC = %0.2f' % roc_auc)
plt.legend(loc = 'lower right')
plt.plot([0, 1], [0, 1],'g--')
plt.xlim([0, 1])
plt.ylim([0, 1])
plt.ylabel('True Positive Rate')
plt.xlabel('False Positive Rate')
plt.show()

In [ ]:
AUC = roc_auc_score(valid_df[target].values, preds)
AUCPR = average_precision_score(valid_df[target].values, preds)

print(f"AUC of {model_name} = ", AUC)
print(f"AUC-PR of {model_name} = ", AUCPR)

AUC, AUCPR


In [ ]:
# D9: operating point at the ALERT_RATE investigation budget
score_at_budget(model_name, valid_df[target].values, preds)

budget_df


In [ ]:
# V24: register K-NN for the one-time test evaluation
assert model_name == "K-NN", model_name
fitted_models[model_name] = lambda d, m=clf: m.predict_proba(d[predictors])[:, 1]
valid_thresholds[model_name] = budget_threshold(preds)


In [ ]:
# Phase 1: this cell used to overwrite AUC with
# roc_auc_score(valid_df[target].values, binary_preds), which for hard labels
# returns (recall + specificity) / 2 rather than an ROC-AUC. Removed.
AUC, AUCPR


In [ ]:
MCC = matthews_corrcoef(valid_df[target].values, binary_preds)
print(f"Matthew's Correlation Coefficient of {model_name} = ", MCC)

bcr = (recall+specificity)/2
print(f"Balanced Classification Rate of {model_name} = ", bcr)

#### Result of All Models

In [ ]:
result_df[model_name] = [recall, precision, specificity, f1, AUC, AUCPR, MCC, bcr]

result_df

In [ ]:
# Phase 1 (Revision_Plan_V01.md, item 5): persist the corrected metrics table.
# V04 was saved by hand; V05 is written here so the re-run is reproducible and
# can be diffed against the submitted numbers.
#
# What changed from models_metric_result_V04.csv:
#   - AUC now comes from predicted probabilities for every model. In V04 six of
#     the eight models passed thresholded 0/1 labels to roc_auc_score, which
#     returns (recall + specificity) / 2, so the AUC column duplicated the
#     Balanced Classification Rate column.
#   - AUC-PR (average precision) added, the metric Reviewer 1 asked for.
#   - XGBoost and LightGBM are scored on valid_df like the other six; in V04
#     they were scored on test_df, and XGBoost additionally at a 0.175
#     threshold rather than 0.5.
#   - "LightGDM" is spelled LightGBM.
# D6/D9: V06 carries the chronological split and the budget operating point.
result_df.to_csv(f"result_csv/models_metric_result_V06_{TAG}.csv", index=False)
budget_df.to_csv(f"result_csv/operating_point_V01_{TAG}.csv", index=False)

result_df


# Guard against the failure mode that corrupted the first V23 run: if a model's
# cell raises, `preds` keeps the previous model's predictions and the scoring
# cells below rescore it under the new name, silently. Two identical columns is
# the signature. LightGBM's Dataset cell failed this way and duplicated XGBoost.
_num = result_df.set_index(result_df.columns[0]).astype(float)
for _a in range(len(_num.columns)):
    for _b in range(_a + 1, len(_num.columns)):
        _ca, _cb = _num.columns[_a], _num.columns[_b]
        assert not _num[_ca].equals(_num[_cb]), (
            f"{_ca} and {_cb} have identical metrics -- one of their cells "
            f"almost certainly failed and reused the other's predictions")
print("column-duplication check passed")


In [ ]:
# ===========================================================================
# V24: the one-time evaluation on the held-out TEST set. These are the numbers
# to report in the paper.
#
# Everything above is scored on valid_df, which by now has been used to choose
# the resampling setting (D7), the hyperparameters (D8), the early-stopping
# round for XGBoost and LightGBM, and the operating threshold (D9). Numbers from
# it are optimistic. test_df -- the chronologically last 20% of transactions --
# has influenced none of those choices and is scored exactly once here, with
# everything frozen: the models as fitted above, and each model's threshold as
# set on the validation set. The achieved alert rate on test will therefore
# drift from ALERT_RATE; that drift is part of the result, not an error.
# ===========================================================================
assert list(fitted_models) == [c for c in result_df.columns[1:]], list(fitted_models)

y_test = test_df[target].values
test_rows, test_budget_rows = {}, {}
for name, predict in fitted_models.items():
    proba = np.asarray(predict(test_df), dtype=float)
    binary = (proba > 0.5).astype(int)
    rec = recall_score(y_test, binary)
    spec = recall_score(y_test, binary, pos_label=0)
    test_rows[name] = [rec, precision_score(y_test, binary, zero_division=0), spec,
                       f1_score(y_test, binary), roc_auc_score(y_test, proba),
                       average_precision_score(y_test, proba),
                       matthews_corrcoef(y_test, binary), (rec + spec) / 2]
    thr, _, usable = valid_thresholds[name]
    if usable:
        alerted = (proba >= thr).astype(int)
        test_budget_rows[name] = [thr, alerted.mean(), recall_score(y_test, alerted),
                                  precision_score(y_test, alerted, zero_division=0),
                                  f1_score(y_test, alerted),
                                  matthews_corrcoef(y_test, alerted)]
    else:
        test_budget_rows[name] = [np.nan] * 6

test_result_df = pd.DataFrame({'metric\\model': list(result_df.iloc[:, 0]), **test_rows})
test_budget_df = pd.DataFrame({'metric\\model': [
    'Operating threshold (from validation)', 'Alert rate on test', 'Recall (budget)',
    'Precision (budget)', 'F1 (budget)', 'MCC (budget)'], **test_budget_rows})

# same guard as the validation export: identical columns mean a model's cell
# failed and its predictions were silently reused
_num = test_result_df.set_index(test_result_df.columns[0]).astype(float)
for _a in range(len(_num.columns)):
    for _b in range(_a + 1, len(_num.columns)):
        assert not _num.iloc[:, _a].equals(_num.iloc[:, _b]), (_num.columns[_a], _num.columns[_b])

test_result_df.to_csv(f"result_csv/test_metric_result_V01_{TAG}.csv", index=False)
test_budget_df.to_csv(f"result_csv/test_operating_point_V01_{TAG}.csv", index=False)
print(f"test set: {len(test_df)} transactions, {int(y_test.sum())} fraudulent")
test_result_df


In [ ]:
# V25 (Revision_Plan_V03.md, D9): precision-recall curves on the test set, with each
# model's operating point at the 0.5% alert budget marked. The curve points are also
# saved, so the base and client-feature runs can be drawn side by side afterwards.
# Colours: the dataviz reference palette in fixed model order (validated; colour
# follows the model, never its rank). Line style is a secondary cue so identity
# never rests on colour alone (colour-blind readers, greyscale print).
from sklearn.metrics import precision_recall_curve
os.makedirs("result_csv/figures", exist_ok=True)
PR_COLOURS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
PR_STYLES = [(0, (1, 1.5)), "-", "-", "--", "--", "-.", ":", "-."]

pr_rows = []
fig, ax = plt.subplots(figsize=(7.2, 5.4))
for (name, predict), colour, style in zip(fitted_models.items(), PR_COLOURS, PR_STYLES):
    proba = np.asarray(predict(test_df), dtype=float)
    prec, rec, _ = precision_recall_curve(y_test, proba)
    keep = np.unique(np.linspace(0, len(rec) - 1, 600).astype(int))   # thin for plotting and storage
    ax.plot(rec[keep], prec[keep], color=colour, ls=style, lw=1.6,
            label=f"{name}  (AUC-PR {average_precision_score(y_test, proba):.3f})")
    pr_rows += [dict(model=name, recall=r, precision=p, point="curve") for r, p in zip(rec[keep], prec[keep])]
    thr, _, usable = valid_thresholds[name]
    if usable:
        alerted = (proba >= thr).astype(int)
        r0, p0 = recall_score(y_test, alerted), precision_score(y_test, alerted, zero_division=0)
        ax.plot(r0, p0, "o", ms=7, color=colour, mec="white", mew=1.4, zorder=5)
        pr_rows.append(dict(model=name, recall=r0, precision=p0, point="operating point (0.5% budget)"))
# the random-ranking baseline goes in the legend: a text label on the plot collides
# with the weak models' curves, which run along the bottom
ax.axhline(y_test.mean(), color="#8a8984", lw=1, ls=(0, (2, 2)),
           label=f"Random ranking  (fraud rate {y_test.mean():.4f})")
ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.grid(color="#e6e5e1", lw=0.6); ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.legend(fontsize=8, frameon=False, loc="center left", bbox_to_anchor=(1.01, 0.5),
          title="Dots: operating point at a 0.5% alert budget", title_fontsize=8)
fig.savefig(f"result_csv/figures/pr_curves_test_{TAG}.png", dpi=300, bbox_inches="tight")
pd.DataFrame(pr_rows).to_csv(f"result_csv/pr_curves_test_{TAG}.csv", index=False)
plt.show()


In [ ]:
# V26 (Phase 7 plan §10, 4C): no cross-model average. The four columns are different quantities
# -- impurity reduction (decision tree, random forest), prediction-value change (CatBoost) and
# total gain (XGBoost), each normalised to sum to 1 within its model -- so their mean has no common
# meaning. The table reports them side by side, ordered by CatBoost for display only; the one
# consistently defined cross-model measure is the permutation importance in
# revision_scripts/explain_compare.py.
feature_df.sort_values('CatBoost', inplace=True, ignore_index=True, ascending=False)
feature_df.to_csv(f"result_csv/feature_rank_table_V06_{TAG}.csv", index=False)
feature_df


In [ ]:
# Decision 2: XGBoost replaces k-NN in the merged LIME table.
df_lime_dt = pd.read_csv(f"result_csv/LIME_Decision Tree_{TAG}.csv")
df_lime_dt.columns = ['features', 'DT']
                  
df_lime_rf = pd.read_csv(f"result_csv/LIME_Random Forest_{TAG}.csv")
df_lime_rf.columns = ['features', 'RF']

df_lime_catboost = pd.read_csv(f"result_csv/LIME_CatBoost_{TAG}.csv")
df_lime_catboost.columns = ['features', 'Catboost']

df_lime_xgb = pd.read_csv(f"result_csv/LIME_XGBoost_{TAG}.csv")
df_lime_xgb.columns = ['features', 'XGBoost']

df_lime_dt['features'] = df_lime_dt['features'].astype(str)
df_lime_rf['features'] = df_lime_rf['features'].astype(str)
df_lime_catboost['features'] = df_lime_catboost['features'].astype(str)
df_lime_xgb['features'] = df_lime_xgb['features'].astype(str)

df_lime_dt.sort_values('features', inplace=True, ignore_index=True)
df_lime_rf.sort_values('features', inplace=True, ignore_index=True)
df_lime_catboost.sort_values('features', inplace=True, ignore_index=True)
df_lime_xgb.sort_values('features', inplace=True, ignore_index=True)

df_lime = pd.concat([df_lime_dt, df_lime_rf, df_lime_catboost, df_lime_xgb], ignore_index=True, axis=1)

display(HTML(df_lime.head().to_html(index=True,header=True)))

df_lime = df_lime.drop(columns=[2,4,6])
df_lime.columns = ['feature_condition', 'Decision Tree', 'Random Forest', 'CatBoost', 'XGBoost']

df_lime

# V24: LIME_All_Models_V08.xlsx was assembled by hand from the old models.
df_lime.to_excel(f"result_csv/LIME_All_Models_V09_{TAG}.xlsx", index=False)
df_lime


In [ ]:
# V26: retired. This cell built Appendix Table A3 from class-count bins hardcoded from the
# submitted run, which never matched the revised LIME intervals (CLAUDE.md). Table A3 is
# built by revision_scripts/lime_table_a3.py from LIME's own discretiser.


In [ ]:
# V26: retired. This cell built Appendix Table A3 from class-count bins hardcoded from the
# submitted run, which never matched the revised LIME intervals (CLAUDE.md). Table A3 is
# built by revision_scripts/lime_table_a3.py from LIME's own discretiser.


In [ ]:
# V26: retired. This cell built Appendix Table A3 from class-count bins hardcoded from the
# submitted run, which never matched the revised LIME intervals (CLAUDE.md). Table A3 is
# built by revision_scripts/lime_table_a3.py from LIME's own discretiser.
